In [1]:
import os
import pandas as pd
import mlflow
import mlflow.sklearn
import s3fs # Importante para ler diretamente do MinIO
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import Pipeline

# 1. Configuração do MinIO (usando a rede interna do Docker)
minio_storage_options = {
    "key": "minioadmin",
    "secret": "minioadmin123",
    "client_kwargs": {
        "endpoint_url": "http://minio:9000"
    }
}

# 2. Conexão com o MLflow dentro do Docker
tracking_uri = os.environ.get("MLFLOW_TRACKING_URI", "http://mlflow:5000")
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("triageai-baseline-oficial")

print(f"Conectado ao MLflow em: {tracking_uri}")

# 3. Leitura dos Dados direto da Camada Gold no MinIO
path_gold = "s3://gold/textos_rag.csv"
print(f"Lendo dados da camada Gold no MinIO: {path_gold}")

df = pd.read_csv(path_gold, storage_options=minio_storage_options)

print(f"Dados carregados! Total de doenças (linhas): {len(df)}")

# O TF-IDF é inteligente o suficiente para ignorar os colchetes e aspas 
# que estão na string "['sintoma1', 'sintoma2']" e pegar só as palavras.
X = df['sintomas']
y = df['diseases']

# Separando em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Execução e Tracking do Experimento
with mlflow.start_run(run_name="regressao_logistica_dataset_gold"):
    
    # Registrando Parâmetros
    max_iter = 150
    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("vectorizer", "TF-IDF")
    mlflow.log_param("dataset_version", "Gold_v1")
    mlflow.log_param("dataset_path", path_gold) # Boa prática: registrar de onde o dado veio

    # Criando o Pipeline (Texto -> Números -> Modelo)
    pipeline = Pipeline([
        ('tfidf', TfidfVectorizer()),
        ('clf', LogisticRegression(max_iter=max_iter, random_state=42))
    ])

    print("Treinando o modelo com os dados do MinIO...")
    pipeline.fit(X_train, y_train)

    print("Avaliando na base de teste...")
    y_pred = pipeline.predict(X_test)
    
    # Calculando as métricas 
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    # Enviando os resultados para o painel do MLflow
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)

    print(f"Métricas Finais -> Acurácia: {acc:.2f} | F1-Score: {f1:.2f}")

    # Salvando o modelo treinado (os arquivos vão direto pro MinIO)
    mlflow.sklearn.log_model(pipeline, "modelo_classificacao")

    print("✅ Sucesso! Modelo treinado e registrado no MLflow.")

Conectado ao MLflow em: http://mlflow:5000
Lendo dados da camada Gold no MinIO: s3://gold/textos_rag.csv
Dados carregados! Total de doenças (linhas): 246945
Treinando o modelo com os dados do MinIO...


/opt/conda/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Avaliando na base de teste...
Métricas Finais -> Acurácia: 0.85 | F1-Score: 0.85


2026/04/09 23:08:40 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


✅ Sucesso! Modelo treinado e registrado no MLflow.
🏃 View run regressao_logistica_dataset_gold at: http://mlflow:5000/#/experiments/1/runs/fdbeaf04e3304388a19ee3232b239953
🧪 View experiment at: http://mlflow:5000/#/experiments/1
